# Pressure from PIV — results walkthrough

This notebook walks up the **validation ladder** of the project:

1. Taylor–Green vortex (exact, unsteady)
2. Kovasznay flow (exact, steady, viscous)
3. The same flows with synthetic PIV errors
4. CFD cylinder wake at Re = 100
5. Parametric studies

It contains **no solver, derivative or metric code**: every cell only calls the package
(`fields`, `piv`, `core`, `analysis`, `studies`).

**`QUICK` flag.** With `QUICK = True` the notebook re-runs *fast* versions of the sweeps
(3 seeds, coarse grids). It writes them to `results/notebook_quick/`, so the full results
are never overwritten. With `QUICK = False` it runs nothing heavy: it loads the CSVs and
figures that `python run_all.py` wrote to `results/`.

Run it with *Kernel → Restart & Run All*, from the `piv_pressure/` folder or its parent.

In [ ]:
QUICK = True   # True: fast re-run into results/notebook_quick/; False: load the full run_all.py results

import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "core").is_dir() else Path.cwd() / "piv_pressure"
sys.path.insert(0, str(ROOT))
if QUICK:  # must be set before the package is imported
    os.environ["PIV_RESULTS_DIR"] = str(ROOT / "results" / "notebook_quick")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from analysis.metrics import convergence_slope, demean, eps_l2, eps_max, error_map
from analysis.plots import FIG_DIR, RESULTS_DIR, error_panel, field_row, plot_series, set_style
from core.pipeline import reconstruct
from fields.analytic import kovasznay, taylor_green
from fields.cfd_loader import cfd_available
from studies.common import make_case, mean_std, read_csv

set_style()
%matplotlib inline
print("results directory used by this notebook:", RESULTS_DIR)

## Step 1 — Taylor–Green vortex: reconstruct, compare, converge

**What is tested.** The whole chain on an exact unsteady Navier–Stokes solution:
$u=\cos x\sin y\,F,\; v=-\sin x\cos y\,F,\; F=e^{-2\nu t}$, with
$p=-\tfrac{\rho}{4}(\cos 2x+\cos 2y)F^2$. It is sampled on the **non-periodic** window
$[0.3, 2.5]^2$, so the boundary conditions matter.

**Equations used.**
$\nabla p = \mathbf G = -\rho\,\dfrac{D\mathbf u}{Dt} + \mu\nabla^2\mathbf u$
(Eulerian: central differences in time and space), then
$\nabla^2 p = \nabla\cdot\mathbf G$ with $\partial p/\partial n = \mathbf n\cdot\mathbf G$ on
all edges (5-point Laplacian, ghost-point Neumann, zero-mean constraint).
Error: $\epsilon_{L2}=\|p'-p'_{ref}\|_2/\|p'_{ref}\|_2$ after removing both means.

In [ ]:
fld = taylor_green(64)
rec = reconstruct(fld, method="poisson")
print(f"eps_L2 = {eps_l2(rec.p, rec.p_ref):.3e},  eps_max = {eps_max(rec.p, rec.p_ref):.3e}")
print(f"relative compatibility residual = {rec.diagnostics['compatibility_relative']:.2e}")

p_ref, p = demean(rec.p_ref), demean(rec.p)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
field_row(axes[:2], fld.x, fld.y, [p_ref, p], ["exact p", "reconstructed p (Poisson, 64²)"],
          cbar_label="p - mean (non-dim.)")
error_panel(axes[2], fld.x, fld.y, error_map(p, p_ref), "|p - p_ref|")
plt.show()

In [ ]:
ns = [16, 32, 64] if QUICK else [16, 32, 64, 128, 256]
hs, errs = [], []
for n in ns:
    f = taylor_green(n)
    r = reconstruct(f, method="poisson")
    hs.append(f.dx)
    errs.append(eps_l2(r.p, r.p_ref))
slope = convergence_slope(hs, errs)
print("h     :", np.round(hs, 4))
print("eps_L2:", [f"{e:.2e}" for e in errs])
print(f"fitted slope = {slope:.2f}")
print("slope between successive grids:", np.round(np.diff(np.log(errs)) / np.diff(np.log(hs)), 2))

fig, ax = plt.subplots(figsize=(4.5, 3.3))
plot_series(ax, hs, errs, "poisson", label=f"Poisson (slope {slope:.2f})")
ax.plot(hs, errs[-1] * (np.array(hs) / hs[-1]) ** 2, "k--", lw=1, label="slope 2")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("grid spacing h (non-dimensional)"); ax.set_ylabel(r"$\epsilon_{L2}$ (-)")
ax.legend(); plt.show()

**What the result shows.** The reconstructed field reproduces the exact pressure: the
saddle-point maximum is in the centre and the lows are at the corners. The error map is
smooth and reaches its maximum at a window corner, where two Neumann conditions and the
one-sided stencils meet. The slope between successive grids rises towards 2 as h shrinks
(the fitted slope over the coarse QUICK grids is a little below 2 because the coarsest grid
is pre-asymptotic). This confirms second-order accuracy of the whole chain: the derivatives,
G, the Neumann boundary conditions and the Poisson solve. The compatibility residual is
small, as expected for exact data.

## Step 2 — Kovasznay flow: side-by-side fields and error map

**What is tested.** A steady exact solution in which the **viscous term** is important:
$u = 1-e^{\lambda x}\cos 2\pi y$, $v=\tfrac{\lambda}{2\pi}e^{\lambda x}\sin 2\pi y$,
$p=\tfrac{\rho}{2}(1-e^{2\lambda x})$, $\lambda = Re/2-\sqrt{Re^2/4+4\pi^2}$, at Re = 20.

**Equations used.** The same pipeline with $\partial\mathbf u/\partial t = 0$, so
$\mathbf G = -\rho(\mathbf u\cdot\nabla)\mathbf u + \mu\nabla^2\mathbf u$. As a check, the
reconstruction is repeated **without** the $\mu\nabla^2\mathbf u$ term.

In [ ]:
kf = kovasznay(64, Re=20)
kr = reconstruct(kf)
kr_inv = reconstruct(kf, include_viscous=False)
print(f"with viscous term:    eps_L2 = {eps_l2(kr.p, kr.p_ref):.3e}")
print(f"without viscous term: eps_L2 = {eps_l2(kr_inv.p, kr_inv.p_ref):.3e}")

pr, pk = demean(kr.p_ref), demean(kr.p)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
field_row(axes[:2], kf.x, kf.y, [pr, pk], ["exact p (Re = 20)", "reconstructed p"],
          cbar_label="p - mean (non-dim.)")
error_panel(axes[2], kf.x, kf.y, error_map(pk, pr), "|p - p_ref|")
plt.show()

**What the result shows.** The two fields share one colour scale and are practically
identical. The pressure rises steeply near the inflow edge x = −0.5, and that is where the
error map peaks. Leaving out the viscous term raises the error by about an order of magnitude
(see the two printed values), so μ∇²u cannot be neglected at these Reynolds numbers.

## Step 3 — Synthetic PIV errors: one noisy example and the noise sweep

**What is tested.** How measurement errors propagate to the pressure. The PIV model
(`piv/degrade.py`) applies, in order:
1. a box filter of width W (interrogation window; here W = h);
2. subsampling to the PIV spacing h;
3. Gaussian noise with standard deviation σ·U added to u and v of every snapshot (seeded).

**Equations used.** The same as step 1. The noise enters Du/Dt through the time difference,
which amplifies it by about σU/(√2 Δt), and through the spatial derivatives, by about σU/h.

In [ ]:
clean = make_case("tg", 64, window_h=1.0, sigma=0.0)
noisy = make_case("tg", 64, window_h=1.0, sigma=0.02, seed=0)
rc, rn = reconstruct(clean), reconstruct(noisy)
print(f"sigma = 0 %: eps_L2 = {eps_l2(rc.p, rc.p_ref):.3e}")
print(f"sigma = 2 %: eps_L2 = {eps_l2(rn.p, rn.p_ref):.3e}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
field_row(axes[:1], noisy.x, noisy.y, [noisy.u[1] - clean.u[1]], ["velocity noise u' (σ = 2 %)"],
          cbar_label="u' / U")
field_row(axes[1:], noisy.x, noisy.y, [demean(rn.p_ref), demean(rn.p)],
          ["exact p", "reconstructed from noisy PIV"], cbar_label="p - mean (non-dim.)")
plt.show()

In [ ]:
noise_csv = RESULTS_DIR / "noise.csv"
if QUICK:
    from studies import run_noise
    run_noise.main(["--quick", "--cases", "tg", "kov20", "cfd"])
elif not noise_csv.is_file():
    raise FileNotFoundError("run `python run_all.py` first (or set QUICK = True)")
rows = read_csv(noise_csv)
for case in sorted({r["case"] for r in rows}):
    print(case)
    for s in sorted({float(r["sigma"]) for r in rows if r["case"] == case}):
        line = "  ".join(f"{m}: {mean_std(rows, 'eps_l2', case=case, method=m, sigma=s)[0]:.3e}"
                         for m in ("poisson", "direct", "omni"))
        print(f"  sigma = {100 * s:4.1f} %  {line}")
display(Image(filename=str(FIG_DIR / "fig03_noise.png"), width=900))

**What the result shows.** Even 2 % noise produces visible small-scale ripples in the
reconstructed pressure. In the sweep the error grows roughly **linearly** with σ (the
fitted log-log slopes are close to 1, see `SUMMARY.md`). The Poisson and omni-directional
methods stay well below single-path direct integration, because they average the
non-conservative (curl) part of the noisy G instead of accumulating it along one path.

## Step 4 — CFD cylinder wake (Re = 100)

**What is tested.** A realistic unsteady wake: the OpenFOAM velocity is sampled on the
81 × 81 window x/D ∈ [1, 5], y/D ∈ [−2, 2] every 0.05 D/U. The pressure is reconstructed
from that velocity and compared with the CFD pressure.

**Equations used.** Same pipeline (Poisson, Eulerian and Lagrangian Du/Dt). The CFD part
first needs the grid-independence table (St, mean Cd, Cl amplitude from the `forceCoeffs`
output), and the vortex-core line is taken through the pressure minimum in x ∈ [2, 4.8].

This step needs the OpenFOAM runs (`sh cfd/run_cases.sh`) and `python cfd/export_window.py`.
Without them it prints what is missing and moves on.

In [ ]:
gi = ROOT / "results" / "cfd" / "grid_independence.md"
if gi.is_file():
    display(Markdown(gi.read_text()))
else:
    print("No grid-independence table yet: run `sh cfd/run_cases.sh` (OpenFOAM) first.")

if cfd_available():
    if QUICK:
        from studies import run_cfd_validation
        run_cfd_validation.main(["--quick"])
    for name in ("fig05_cfd_wake", "fig06_vortex_core_line", "fig05b_cfd_error_history"):
        path = FIG_DIR / f"{name}.png"
        if path.is_file():
            display(Image(filename=str(path), width=900))
else:
    print("CFD window export not found - CFD comparison skipped.")

**What the result shows** (when the CFD data are present): the table shows how St, mean
Cd and the Cl amplitude change from the coarse to the fine mesh, and compares them with
the published Re = 100 values (marked [verify]). The wake figure puts the CFD and
reconstructed pressure on one colour scale, and the error map shows where the
reconstruction departs. The line plot through a vortex core compares the depth of the
suction peak. `results/SUMMARY.md` lists the numbers.

## Step 5 — Parametric studies

**What is tested.** One parameter at a time around the baseline
(n = 64, W = h, σ = 1 %, Δt = 0.05, Poisson, all-Neumann):
grid spacing h, noise σ, time step Δt (Eulerian vs Lagrangian), boundary conditions,
integration method, window size W and denoising.

**Equations used.** As above. The Lagrangian scheme uses
$D\mathbf u/Dt \approx [\mathbf u(\mathbf x+\mathbf u\Delta t, t+\Delta t) - \mathbf u(\mathbf x-\mathbf u\Delta t, t-\Delta t)]/(2\Delta t)$.
Bernoulli edges use $p = p_\infty + \tfrac12\rho(U_\infty^2-|\mathbf u|^2)$.

In [ ]:
from analysis import summary
if QUICK:
    from studies import run_bc, run_convergence, run_denoise, run_dt, run_method, run_window
    run_convergence.main(["--quick"])
    run_dt.main(["--quick"])
    run_bc.main(["--quick"])
    run_method.main(["--quick"])
    run_window.main(["--quick"])
    run_denoise.main(["--quick"])
summary.main(RESULTS_DIR)
for name in ("fig01_tg_convergence", "fig02_kovasznay_fields", "fig07_dt", "fig08_bc_method",
             "fig09_window_denoise"):
    path = FIG_DIR / f"{name}.png"
    if path.is_file():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path), width=900))

**What the results show** (the numbers are in `SUMMARY.md` above; every sentence there is
generated from the CSVs):

* **Grid spacing**: on clean data the error falls as h² (fitted slopes ≈ 2). With noise it
  stops falling, and the table reports the optimum spacing.
* **Time step**: for the *convected* vortex and the CFD wake the time step matters, and
  the table shows which scheme degrades faster. For the stationary vortex ∂u/∂t ≈ 0, so the
  Eulerian scheme is almost insensitive to Δt. That is the control case.
* **Boundary conditions**: a Bernoulli edge helps only where the flow really is steady and
  irrotational, which is not true anywhere in the analytic cases. Exact data on two opposite
  edges help, while a single Dirichlet edge can make things worse.
* **Methods**: Poisson and omni-directional integration are far less path-dependent than
  direct integration.
* **Window W**: larger windows lower the vortex-core suction (the core ratio drops below 1).
* **Denoising**: it hurts on clean data and helps at high noise.

# Convergence and error analysis (ME-415, Lecture 11)

This section follows the order of the lecture: **iteration residuals → norms → scaled
residuals → steady-state criterion → grid convergence → CFD grid independence**.
Every iterative solver below works on exactly the same discrete Poisson system as the
default direct solver (`spsolve`), so only the solution method changes.

## 6.1 Iteration errors: the residual

**What is tested.** How the residual $\mathbf r = \mathbf b - A\mathbf p^k$ of the
pressure-Poisson system falls with iteration k for Jacobi, Gauss–Seidel, SOR and conjugate
gradients (Taylor–Green, all-Neumann, zero mean). The solve is converged when
$\|\mathbf r\| < \mathrm{tol}$.

**Equations used.**
- Jacobi: $\mathbf p^{k+1} = \mathbf p^k + \omega D^{-1}(\mathbf b - A\mathbf p^k)$.
- SOR: the same update node by node (red, then black), using the newest values; $\omega = 1$
  gives Gauss–Seidel.
- CG: Krylov iterations on the symmetric form $-WA$.

In [ ]:
if QUICK:
    from studies import run_residuals
    run_residuals.main(["--quick"])
hist = read_csv(RESULTS_DIR / "residual_history.csv")
for solver in dict.fromkeys(r["solver"] for r in hist):
    last = [r for r in hist if r["solver"] == solver][-1]
    print(f"{solver:15s} iterations {last['iteration']:>7s}  time {float(last['time_s']):7.3f} s  "
          f"converged {last['converged']:5s}  final scaled residual {float(last['residual_scaled']):.2e}")
display(Image(filename=str(FIG_DIR / "fig11_residual_history.png"), width=950))

**What the result shows.**
- All four methods drive the residual down to the tolerance, but at very different costs:
  SOR and CG need tens to hundreds of iterations, while Jacobi and Gauss–Seidel need
  thousands (Gauss–Seidel about half as many as Jacobi).
- *Classic* Jacobi ($\omega = 1$) levels off above the tolerance. The all-Neumann problem has
  a checkerboard mode that classic Jacobi never damps. The lecture's remark applies: what
  matters is that the residual curve levels off at a low value. The default $\omega = 0.95$
  removes the stall.

## 6.2 Norms

**What is tested.** The same residual measured in different norms, and why the normalised
forms are needed.

**Equations used.**
- $L_1 = \sum|r_i|$, $L_2 = \sqrt{\sum r_i^2}$, $L_\infty = \max|r_i|$.
- $L_{1n} = \tfrac1n\sum|r_i|$, $L_{2n} = \sqrt{\tfrac1n\sum r_i^2}$.

In [ ]:
for r in read_csv(RESULTS_DIR / "norm_scaling.csv"):
    print(f"n = {r['n']:>4s}: L1 {float(r['l1']):9.3g}  L2 {float(r['l2']):9.3g}  "
          f"L1n {float(r['l1n']):7.3g}  L2n {float(r['l2n']):7.3g}")
display(Image(filename=str(FIG_DIR / "fig12_norm_comparison.png"), width=950))

**What the result shows.**
- (a) The three norms of one residual follow the same trend. $L_\infty$ sits highest because
  it sees only the worst cell; $L_{1n}$ sits lowest because it weights all cells equally.
- (b) For the same physical problem, the unnormalised $L_1$ and $L_2$ grow with the number of
  cells (about $N^1$ and $N^{1/2}$), while $L_{1n}$ and $L_{2n}$ stay flat. A fixed tolerance
  on an unnormalised norm would therefore mean something different on every grid.
- The dotted line shows that the ghost-point boundary rows, which contain flux terms $2G/h$,
  make even $L_{2n}$ grow slightly.

## 6.3 Scaled residuals

**What is tested.** Dividing the residual by a characteristic scale S gives a relative
("percentage") imbalance per cell, $r_s = r/S$. Two scales are compared:
- $S = L_{2n}(\mathbf b)$, the typical source magnitude, which plays the role of the flux
  through a face;
- $S = \rho U^2/L^2$, the fundamental pressure-Laplacian scale.

**Equation used.** Stop when $L_{2n}(\mathbf r)/S < \mathrm{tol}$.

In [ ]:
fld = taylor_green(32 if QUICK else 64)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
for kind, ls in (("rhs", "-"), ("physical", "--")):
    h = reconstruct(fld, solver="sor", residual_scale=kind).extra["history"]
    ax.semilogy(h.residual_scaled, ls=ls, label=f"S = {kind}: S = {h.scale:.3g}, {h.iterations} it.")
ax.axhline(1e-8, color="0.4", ls=":", label="tol = 1e-8")
ax.set_xlabel("iteration k"); ax.set_ylabel("scaled residual L2$_n$(r)/S (-)")
ax.legend(fontsize=8); plt.show()

**What the result shows.** The two curves have the same shape and are shifted by the ratio
of the two scales, so the same tolerance ends the iteration at a slightly different
iteration. What matters is not the exact value but that the residual history falls steadily
and levels off at a low value, several orders below its start.

## 6.4 Steady-state criterion $\|u^{n+1}-u^n\| < \varepsilon$

**What is tested.** When the time (or pseudo-time) loop can stop.
- Jacobi is exactly explicit pseudo-time marching of $\partial p/\partial\tau = \nabla^2 p - f$
  with $\Delta\tau = \omega h^2/4$. Its change per step is
  $\|\mathbf p^{k+1}-\mathbf p^k\| = \Delta\tau\,\|\mathbf r^k\|$, and $\Delta\tau$ is tiny.
- For the CFD wake, see Figure 17: the Re = 100 flow is periodic, so a periodicity criterion
  replaces the steady-state one.

**Equation used.** Stop the loop when $L_{2n}(\mathbf p^{k+1}-\mathbf p^k) < \varepsilon$,
and compare that stopping point with the residual and with the true error
$\|\mathbf p^k - \mathbf p^*\|$.

In [ ]:
display(Image(filename=str(FIG_DIR / "fig13_residual_vs_change.png"), width=700))
fig17 = FIG_DIR / "fig17_cfd_residuals.png"
if fig17.is_file():
    display(Image(filename=str(fig17), width=950))
else:
    print("CFD residual figure not available (needs the OpenFOAM runs).")

**What the result shows.** The change per iteration falls below $\varepsilon$ long before the
residual and the true error are small. A small change only means the iterate moves slowly,
which it does when convergence is slow, as it is for Jacobi on fine grids. The change norm
is a safe stopping criterion only together with a residual check.

For the periodic CFD wake, $\|u^{n+1}-u^n\|$ never goes to zero. The cycle-to-cycle change of
the lift amplitude and period is used instead. If the optional steady Re = 20 case was run,
its change norm falls through $\varepsilon$ and OpenFOAM stops the run automatically.

## 6.5 Grid convergence (first vs second order)

**What is tested.** How the pressure error falls as the grid is refined, for the default
second-order scheme and a deliberately first-order one (forward differences, first-order
Neumann boundary conditions).

**Equation used.** With error $\approx C\,\Delta x^{p}$, we have
$\log(\text{error}) = \log C + p\,\log\Delta x$, so the slope on log–log axes is the order p.
The accuracy target is an error below 1 %.

In [ ]:
if QUICK:
    from studies import run_order
    run_order.main(["--quick"])
for r in read_csv(RESULTS_DIR / "order_summary.csv"):
    ext = " (extrapolated)" if r["cross_extrapolated"] == "True" else ""
    print(f"{r['case']:6s} order {r['order']}: slope {float(r['slope_fine']):.2f}, "
          f"error < 1 % for dx < {float(r['dx_cross']):.3g}{ext}")
for name in ("fig16_grid_convergence_tg", "fig16b_grid_convergence_kov20"):
    display(Image(filename=str(FIG_DIR / f"{name}.png"), width=750))

**What the result shows.**
- The clean curves become straight lines, parallel to the slope-1 and slope-2 triangles.
  With `QUICK = True` the grids stop at 64², where the coarse end is still pre-asymptotic, so
  the fitted slopes come out a little below 1 and 2. The full run (`QUICK = False`, grids to
  256²) fits over the asymptotic range and gives slopes much closer to 1 and 2 (see
  `SUMMARY.md` §10).
- The second-order scheme reaches the 1 % target on a much coarser grid. The first-order
  scheme needs a far finer grid for the same accuracy. When a scheme does not reach 1 %
  within the tested grids, the crossing is extrapolated with its fitted slope and marked
  "(extrapolated)".
- With 1 % velocity noise (dashed) the error stops falling and then rises on fine grids,
  because differentiation amplifies noise as $1/\Delta x$ (and the viscous term as
  $1/\Delta x^2$).

## 6.6 CFD grid independence

**What is tested.** Whether St and the mean $C_D$ of the Re = 100 wake converge with mesh
refinement.

**Equations used.** Richardson extrapolation (observed order p, extrapolated value) and the
Grid Convergence Index $\mathrm{GCI} = 1.25\,|e|/(r^p-1)$ of the medium and fine meshes.

In [ ]:
gi = ROOT / "results" / "cfd" / "grid_independence.md"
fig18 = FIG_DIR / "fig18_cfd_grid_convergence.png"
if gi.is_file():
    display(Markdown(gi.read_text()))
if fig18.is_file():
    display(Image(filename=str(fig18), width=900))
else:
    print("CFD grid-independence data not available (run `sh cfd/run_cases.sh`).")

**What the result shows** (when the CFD data exist):
- the % change between meshes should shrink from coarse→medium to medium→fine;
- the observed order should be close to the formal order of the scheme (about 2);
- a fine-mesh GCI of a few percent or less means the fine mesh is grid independent for that
  quantity;
- an asymptotic ratio close to 1 means the three meshes are in the asymptotic range.

Literature values are marked [verify].

In [ ]:
summary.main(RESULTS_DIR)
display(Markdown((RESULTS_DIR / "SUMMARY.md").read_text()))